# Лабораторная работа №1  
## Данные для систем ИИ: исследование, качество и подготовка

**Цель:** пройти путь от «сырого» табличного набора данных до набора, пригодного для последующего машинного обучения.

Файл данных: `ai_clients_lab1.csv`.

Выполняйте задания по порядку. Не удаляйте исходный `df`: для очистки создайте копию `clean_df`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option('display.max_rows', None)
pd.set_option("display.precision", 2)

df = pd.read_csv("ai_clients_lab1.csv")
df.head()


### Задание 1. Первичное знакомство
Получите размер набора данных, список столбцов, типы данных, первые и последние строки.  
Ответьте: что является объектом наблюдения? Какие признаки числовые, категориальные и идентификационные? Какой столбец можно рассматривать как будущую целевую переменную?


In [ ]:
"""
Объект: Пользователь у которого есть подписка на какой-то сервис
"""
print("")
df.count()

print(f"размерность данных: {df.shape=}")

print("Названия столбцов:")
[print(column) for column in df.columns]


# print("общая информация:")
# print(df.info())

print("типы столбцов:")
print(df.dtypes)

print("первые 5 строк:")
print(df.head())
print("последние 5 строк:")
print(df.tail())


### Задание 2. Качество данных
Найдите:
- пропущенные значения по столбцам;
- полные дубликаты строк;
- некорректные значения возраста;
- необычно большие значения `MonthlySpend` и `UsageHours`.

Не исправляйте данные, пока не зафиксируете найденные проблемы.


In [64]:
# print("Пропущенные значения по столбцам")
# print(df.isnull().sum())

# print("Полные дубликаты строк")

# duplicates = df[df.duplicated()]
# display(duplicates)
# print("Кол-во полных дубликатов:", df.duplicated().sum())

# print("некорректные значения возраста:")
# invalid_age = df[(df['Age'] < 0) | (df['Age'] > 100)]

# if len(invalid_age) > 0:
#     display(invalid_age[['CustomerId', 'Age']])

print("Вот тут я пытался самостоятельно найти средние, и как-то по ним вычислять аномально большие значения:")

print("средние значения:")
MonthlySpendAVG = df["MonthlySpend"].mean()
UsageHoursAVG = df["UsageHours"].mean()
display(MonthlySpendAVG)
display(UsageHoursAVG)

print("медианные значения:")
MonthlySpendMED = df["MonthlySpend"].median()
UsageHoursMED = df["UsageHours"].median()
display(MonthlySpendMED)
display(UsageHoursMED)

print("Моя теория выдавала слишком много значений:")


big_MonthlySpend = df[df["MonthlySpend"] > MonthlySpendAVG]
display(big_MonthlySpend.head(n=7))

print("Сортировкой вроде даже что-то находилось:")

display(df.sort_values(by="MonthlySpend", ascending=False).head(n=7))

print("Но отделить то как? Должен ведь универсальный метод быть, и я полез искать методы")
print("Нашёл и почитал про квадрат Тьюки и межквартильный размах, и применил его:")

# MonthlySpend
Q1 = df['MonthlySpend'].quantile(0.25)
Q3 = df['MonthlySpend'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR
unusual_spend = df[df['MonthlySpend'] > upper]
display(unusual_spend[['MonthlySpend']])

print("Хоть это и эмпирически верное решение, но я подумал что коэффициент лучше докрутить до 1.7 минимум, потому что я не считаю что 2657.96 или 2615.62 это анимально большие значения")

print("И да, наверное в анализе данных или ml потеря условно пары верхних значений не критична, поэтому берут 1.5, но 1.7 в данном контексте дало более точные результаты")

print("Создал метод специальный для этого")

def interquartile(df, by_row: str):
    _df = df
    Q1 = _df[by_row].quantile(0.25)
    Q3 = _df[by_row].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.7 * IQR
    unusual_spend = _df[_df[by_row] > upper]
    display(unusual_spend[[by_row]])

print('итого:')

interquartile(df=df, by_row='MonthlySpend')
interquartile(df=df, by_row='UsageHours')

Вот тут я пытался самостоятельно найти средние, и как-то по ним вычислять аномально большие значения:
средние значения:


np.float64(1496.6924749163882)

np.float64(32.63933884297521)

медианные значения:


np.float64(1465.76)

...

Но отделить то как? Должен ведь универсальный метод быть, и я полез искать методы
Нашёл и почитал про квадрат Тьюки и межквартильный размах, и применил его:


,MonthlySpend
55,5200.00
184,2657.96
219,6800.00
377,7500.00
445,2615.62
514,9100.00


Хоть это и эмпирически верное решение, но я подумал что коэффициент лучше докрутить до 1.7 минимум, потому что я не считаю что 2657.96 или 2615.62 это анимально большие значения
И да, наверное в анализе данных или ml потеря условно пары верхних значений не критична, поэтому берут 1.5, но 1.7 в данном контексте дало более точные результаты
Создал метод специальный для этого
итого:


,MonthlySpend
55,5200.0
219,6800.0
377,7500.0
514,9100.0


,UsageHours
81,83.2
88,170.0
100,82.8
130,79.6
136,96.9
251,78.0
278,98.5
314,77.8
322,93.3
340,75.3


### Задание 3. Очистка
Создайте `clean_df = df.copy()` и:
1. удалите полные дубликаты;
2. некорректный возраст замените на `NaN`, затем заполните пропуски медианой;
3. пропуски `Region` заполните наиболее частой категорией;
4. пропуски `MonthlySpend` заполните медианой;
5. выбросы `MonthlySpend` определите по правилу IQR и удалите только верхние экстремальные значения;
6. для `UsageHours` определите выбросы и примите обоснованное решение: удалить, ограничить или оставить.

После очистки проверьте данные повторно.


In [ ]:
# Ваш код


### Задание 4. Одномерный анализ
Постройте:
- гистограмму возраста;
- гистограмму месячных расходов;
- столбчатую диаграмму тарифных планов;
- boxplot для `MonthlySpend`.

Для каждого графика сформулируйте 1–2 наблюдения.


In [ ]:
# Ваш код


### Задание 5. Связи между признаками
Исследуйте связь `Churn` минимум с тремя признаками. Обязательно включите:
- `AutoPay`;
- `Plan`;
- один числовой признак по вашему выбору.

Для категориальных признаков используйте `pd.crosstab(..., normalize='index')`. Для числового признака сравните группы через `groupby`.


In [ ]:
# Ваш код


### Задание 6. Корреляции
Выберите числовые признаки и вычислите корреляционную матрицу. Визуализируйте её средствами Matplotlib (`imshow`).  
Укажите две наиболее заметные связи и объясните, почему корреляция сама по себе не доказывает причинность.


In [ ]:
# Ваш код


### Задание 7. Feature engineering
Создайте два новых признака:
- `TenureGroup`: `Новый` (1–12 мес.), `Стабильный` (13–36), `Долгосрочный` (37+);
- `HighSupport`: 1, если обращений в поддержку 4 и больше, иначе 0.

Проверьте долю `Churn="Да"` в новых группах.


In [ ]:
# Ваш код


### Задание 8. Подготовка X и y
Создайте:
- `y` из столбца `Churn`, преобразовав `Да/Нет` в `1/0`;
- `X` без `CustomerId` и `Churn`;
- категориальные признаки преобразуйте через `pd.get_dummies(..., drop_first=True)`.

Проверьте размерность `X` и наличие пропусков. **Модель пока не обучаем.**


In [ ]:
# Ваш код


### Задание 9. Сохранение результата
Сохраните очищенный набор как `ai_clients_clean.csv`. В выводе укажите исходное и итоговое число строк.


In [ ]:
# Ваш код


## Итоговый вывод
Ответьте кратко:
1. Какие проблемы качества данных были обнаружены?
2. Какие решения по очистке вы приняли и почему?
3. Какие признаки визуально/аналитически связаны с оттоком?
4. Почему нельзя по этим наблюдениям утверждать, что найденные признаки **причинно** вызывают отток?
5. Почему подготовка данных является частью разработки системы ИИ, а не отдельной «технической» процедурой?
